# Stage-z drift QC

Every acquired movie writes a HAL `.off` focus-lock log alongside its image file (same directory, same stem). It's a whitespace-delimited table, one row per frame, with columns `frame offset power stage-z good-offset`. The focus lock is expected to hold `stage-z` constant for a whole FOV's stack.

This notebook, for every round and every FOV:
- reads the `.off` sidecar's `stage-z` column (tolerating a sidecar that HAL has created but not finished writing yet -- e.g. when this notebook is run *during* acquisition -- by skipping it and picking it up on a later run, rather than crashing),
- checks whether every row is the same value,
- if not, records the first row's value plus the min/max (should be rare),
- caches the result to `analysis/stage_z_summary.csv` so re-running this notebook only reads *new* `.off` files, never re-reading ones already summarized,
- plots the first-frame `stage-z` value against `fov_id`, one line per round, all overlaid on the same x-axis (so every round shares the same FOV x-position and round-to-round drift at a given FOV shows up as a vertical offset between lines, not a left-to-right shift).

This is a one-shot notebook — re-run any of its cells at any point during or after acquisition to refresh the plot with whatever has been written so far.

## 1 — Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/analysis/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.analysis.stage_z import update_stage_z_cache, assign_x_positions, round_label

print(f"SAMPLE_DIR : {SAMPLE_DIR}")

## 2 — Experiment parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
IMAGE_SUFFIX = ".zarr"   # must match what HAL is writing

config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = SAMPLE_DIR / "metadata",
    analysis_dir   = SAMPLE_DIR / "analysis",
    settings_dir   = SAMPLE_DIR / "settings",
    round_info_csv = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
)

meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt,
                                config.data_dir, image_suffix=config.image_suffix)

CACHE_PATH = config.analysis_dir / "stage_z_summary.csv"

print(f"Sample name  : {SAMPLE_NAME}")
print(f"Positions tag: {POSITIONS_TAG}")
print(f"Rounds : {meta.n_rounds}")
print(f"FOVs   : {meta.n_fovs}")
print(f"Cache  : {CACHE_PATH}")

## 3 — Read / update the stage-z cache

Re-run this cell any time — it only reads `.off` files not already in the cache.

In [ ]:
n_before = 0
if CACHE_PATH.exists():
    import pandas as pd
    n_before = len(pd.read_csv(CACHE_PATH))

cache = update_stage_z_cache(config, meta, CACHE_PATH)
print(f"Stage-z cache: {len(cache)} FOV\u00d7series entries total "
      f"({len(cache) - n_before} newly read this run).")

drifted = cache[~cache["all_same"]]
if drifted.empty:
    print("Every FOV's stage-z was constant across its whole frame stack.")
else:
    print(f"{len(drifted)} FOV(s) had a non-constant stage-z within their stack:")
    display(
        drifted[["round_id", "fov_id", "series", "first_stage_z", "min_stage_z", "max_stage_z"]]
        .sort_values(["round_id", "fov_id"])
    )

## 4 — Plot drift over the acquisition

One line per round (cells, hyb01, hyb02, ...), each plotted at the SAME x positions (`fov_id`) and overlaid in a different color, so drift at a given FOV across rounds reads as a vertical spread between lines rather than a shift along a long concatenated axis.

In [ ]:
plot_df = assign_x_positions(cache)

fig, ax = plt.subplots(figsize=(11, 5))

round_ids = sorted(plot_df["round_id"].unique())
cmap      = plt.cm.turbo
color_of  = {rid: cmap(i / max(len(round_ids) - 1, 1)) for i, rid in enumerate(round_ids)}

for rid in round_ids:
    sub = plot_df[plot_df["round_id"] == rid].sort_values("fov_id")
    ax.plot(sub["x"], sub["first_stage_z"], "-o", ms=1, lw=.5, alpha=0.8,
            color=color_of[rid], label=round_label(meta, rid))

ax.set_xlabel("FOV id")
ax.set_ylabel("stage-z (µm) — first frame of each FOV")
ax.set_title("Stage-z drift across the acquisition (every round overlaid at the same FOV x-position)")
ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1), borderaxespad=0)
ax.set_ylim([0, 100])  # MF3 nanopositioner's full travel range
fig.subplots_adjust(right=0.78)
plt.show()

## 5 — Spatial stage-z heatmap, per round

Maps each FOV's stage (x, y) position to an integer grid index (same
approach as `04_view_intensity_stats.ipynb`'s / `measure_tissue_thickness_test.ipynb`'s
heatmaps: round to the nearest integer micron, then rank each axis's unique
values — robust to float imprecision on a regular grid), fills a matrix with
that FOV's `first_stage_z`, and displays it with a diverging `RdBu` colormap
so a stage tilt or local warp reads as a spatial gradient rather than just a
per-FOV number. `rounds_to_display` picks which rounds (by `round_label`) get
a heatmap — edit the list to add more.

In [ ]:
def positions_to_grid_indices(fov_ids, meta):
    """Stage (x, y) positions -> integer (x_idx, y_idx) grid indices (same approach as
    04_view_intensity_stats.ipynb's / measure_tissue_thickness_test.ipynb's heatmaps:
    round to the nearest integer micron, then rank each axis's unique values --
    robust to float imprecision on a regular grid)."""
    xs = np.array([round(meta.fovs[f].position[0]) for f in fov_ids])
    ys = np.array([round(meta.fovs[f].position[1]) for f in fov_ids])
    unique_xs = np.sort(np.unique(xs))
    unique_ys = np.sort(np.unique(ys))
    x_rank    = {v: i for i, v in enumerate(unique_xs)}
    y_rank    = {v: i for i, v in enumerate(unique_ys)}
    return {f: (x_rank[xs[i]], y_rank[ys[i]]) for i, f in enumerate(fov_ids)}


rounds_to_display = ['cells', 'hyb01']   # round labels (see round_label) to show heatmaps for

display_round_ids = [
    rid for rid in sorted(cache["round_id"].unique())
    if round_label(meta, rid) in rounds_to_display
]

n_panels = max(len(display_round_ids), 1)
fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 5.5), squeeze=False)
axes = axes[0]

for ax, round_id in zip(axes, display_round_ids):
    label   = round_label(meta, round_id)
    sub     = cache[cache["round_id"] == round_id]
    fov_ids = sub["fov_id"].tolist()

    grid = positions_to_grid_indices(fov_ids, meta)
    n_x  = max(xi for xi, _ in grid.values()) + 1
    n_y  = max(yi for _, yi in grid.values()) + 1

    matrix = np.full((n_y, n_x), np.nan)
    for fov_id, z in zip(sub["fov_id"], sub["first_stage_z"]):
        xi, yi = grid[fov_id]
        matrix[yi, xi] = z

    im = ax.imshow(matrix, cmap="RdBu", origin="upper")
    ax.set_title(f"Stage-z heatmap — {label}")
    ax.set_xlabel("X grid index  (increasing stage X →)")
    ax.set_ylabel("Y grid index  (increasing stage Y ↓)")
    fig.colorbar(im, ax=ax, label="stage-z (µm)")

fig.tight_layout()
plt.show()